# 02 — Label Construction

**Workstream**: EDA / labels  ·  **Owner**: Aurelia (backup: Bella)  ·  **Last touched**: 2026-06-08

**What this notebook decides**

Construct the **prediction target** for the risk model from the raw
Chicago Food Inspections data. The label answers:

> *Given a restaurant's inspection on date D, will it receive a* **Fail**
> *or a* **priority violation (code 1–29)** *within the next 180 days?*

**Why a forward window?** A model that consumes the inspection-date snapshot
and predicts the next-window outcome reflects how this would deploy in
production: a city inspector or consumer-facing app would ask "what's the
risk *now*?" — without yet knowing today's inspection result. The 180-day
window matches the median Chicago re-inspection cadence (~220-300 days,
see notebook 01) without being so short that most rows are right-censored.

**Why this notebook is short.** All of the label semantics live in
[`src/foodsafety/data/labels.py`](../src/foodsafety/data/labels.py). The
tests in `tests/test_labels.py` pin down the leak-free window (anchor
excluded; upper bound inclusive at 180 days, exclusive at 181), the burn-in
rule (pre-2019 rows get NA labels, never 0), and the cross-license
isolation. This notebook is the consumer of that module — it calls
`build_labels`, validates the result, and writes the contract artifact
`data/processed/inspections_labeled.parquet`.

**Contract**: `docs/interface_contracts.md` § 1.  
**Scope**: `CLAUDE.md` § "What is IN scope" — label window 180 d, train cutoff 2019-01-01.

**Deliverable**: `data/processed/inspections_labeled.parquet`.

## 1. Setup

Add the project's `src/` to `sys.path` so the package imports work without
having to install. When the team standardises on `uv sync`, this cell can be
trimmed to plain imports.

In [ ]:
import sys
from pathlib import Path

# Notebook lives in notebooks/; package lives in src/foodsafety/. Add ../src.
# Find the repo root by searching upward from the current working directory.
# This handles VS Code notebook kernels running from unexpected cwd values.
def find_project_root(start: Path) -> Path:
    current = start.resolve()
    while True:
        if (current / 'src' / 'foodsafety' / 'config.py').exists():
            return current
        if current.parent == current:
            raise RuntimeError('Could not locate project root from notebook CWD.')
        current = current.parent

_PROJECT_ROOT = find_project_root(Path.cwd())
if str(_PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT / 'src'))

import pandas as pd
import matplotlib.pyplot as plt

from foodsafety.config import (
    PROCESSED_DIR, RAW_DIR, LABEL_WINDOW_DAYS, TRAIN_START_DATE,
)
from foodsafety.data.labels import build_labels, add_violation_features

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

## 2. Load raw inspections

Reads the cache produced by notebook 01. If the file is missing, run
notebook 01 first (it pulls from the Chicago SODA API).

In [ ]:
inspections_path = RAW_DIR / 'inspections.parquet'
if not inspections_path.exists():
    raise SystemExit(
        f'Missing {inspections_path}. Run notebooks/01_dataset_overview_eda.ipynb first.'
    )

raw = pd.read_parquet(inspections_path)
raw['inspection_date'] = pd.to_datetime(raw['inspection_date'])

# Drop the Socrata-internal nested location column and any computed_region
# columns. These are noise and the latter aren't portable across snapshots.
raw = raw.loc[:, ~raw.columns.str.startswith(':@computed_region')]
if 'location' in raw.columns:
    raw = raw.drop(columns=['location'])

print(f'Loaded {len(raw):,} inspections from {inspections_path}')
print(f'Date range: {raw["inspection_date"].min().date()} → {raw["inspection_date"].max().date()}')
print(f'Modified:   {pd.Timestamp(inspections_path.stat().st_mtime, unit="s")}')

## 3. What are we labeling?

Before running the label builder, ground-truth the input distribution. Two
quantities we expect to see in the labels later:

- **Per-row event rate** = P(result = Fail OR violations include code 1–29).
  This is what we aggregate forward over 180 days.
- **Modelable share** = fraction of rows where `results ∈ {Pass, Pass w/ Conditions, Fail}`.
  Other values (Out of Business, No Entry, Not Ready, Business Not Located)
  describe an inspector's failure to perform the inspection. Per the contract,
  they remain in `inspections_labeled.parquet` but feature engineering
  drops them before training.

In [ ]:
print('--- results distribution ---')
print(raw['results'].value_counts(dropna=False).to_string())

modelable_share = raw['results'].isin({'Pass', 'Pass w/ Conditions', 'Fail'}).mean()
print(f'\nModelable share: {modelable_share:.1%}')

# Quick look at the per-row event rate. Uses the same regex as labels.py.
from foodsafety.data.labels import has_priority_violation
raw_event = (raw['results'] == 'Fail') | raw['violations'].apply(has_priority_violation)
print(f'\nPer-row event rate (Fail OR priority violation): {raw_event.mean():.2%}')

### Raw event component breakdown

The target is based on a future `Fail` or priority violation event. Before looking forward in time, this check separates the two components on the raw inspection rows: failures from the `results` column and priority violations parsed from the `violations` text. This helps confirm that the label is not relying on only one source of signal.

In [ ]:
# Break out the two raw event components before constructing the forward label.

raw_event_components = pd.Series({
    "rows_total": len(raw),
    "fail_rows": raw["results"].eq("Fail").sum(),
    "priority_violation_rows": raw["violations"].apply(has_priority_violation).sum(),
    "fail_or_priority_rows": raw_event.sum(),
    "fail_rate": raw["results"].eq("Fail").mean(),
    "priority_violation_rate": raw["violations"].apply(has_priority_violation).mean(),
    "fail_or_priority_rate": raw_event.mean(),
})

raw_event_components.to_frame("value")

## 4. Run the label builder

`build_labels` adds these columns to a copy of the input:

- `is_fail_or_priority` — per-row event flag (the building block, not the label)
- `is_burnin` — True if `inspection_date < TRAIN_START_DATE` (2019-01-01)
- `right_truncated` — True if the 180-day forward window extends past the
  dataset's last observed inspection
- `y_fail_or_critical_next_180d` — the label; **NA** on burn-in rows and on
  rows with placeholder license tokens ("0" or "")

It also renames `license_` → `license_id` to match the cross-team contract.
Per-row event detection uses the regex `(?:^|\|)\s*(\d{1,2})\.\s` on the
`violations` text — pinned down in `tests/test_labels.py`.

Wall-clock: ~30–60 s on 310k rows / ~48k licenses (Python loop, intentionally
kept simple over a vectorised version because correctness is easier to read).

In [ ]:
%%time
labeled = build_labels(
    raw,
    label_window_days=LABEL_WINDOW_DAYS,    # 180
    train_start_date=TRAIN_START_DATE,      # '2019-01-01'
)
print(f'\nshape: {labeled.shape[0]:,} rows × {labeled.shape[1]} cols')

## 5. Validate the labels

Three checks that would catch most kinds of bugs in this kind of pipeline.

### 5a. Burn-in and invalid-license rates

In [ ]:
burnin_n = labeled['is_burnin'].sum()
invalid_lic_n = labeled['license_id'].isin({'0', ''}).sum()
na_label_n = labeled['y_fail_or_critical_next_180d'].isna().sum()

print(f'Burn-in rows (pre-{TRAIN_START_DATE}):     {burnin_n:>8,}  ({burnin_n/len(labeled):.1%})')
print(f'Invalid license tokens ("0" or ""):  {invalid_lic_n:>8,}  ({invalid_lic_n/len(labeled):.1%})')
print(f'NA labels (union of the above):      {na_label_n:>8,}  ({na_label_n/len(labeled):.1%})')
print(f'Right-truncated (last 180 d of data):{labeled["right_truncated"].sum():>8,}  ({labeled["right_truncated"].mean():.1%})')

### 5b. Label prevalence (the headline number)

Healthy class balance is roughly 10–40 % positive. Very low (<5 %) implies
the model will need careful class-imbalance handling; very high (>50 %) is
suspicious and usually means a leak.

In [ ]:
trainable = labeled.dropna(subset=['y_fail_or_critical_next_180d']).copy()
print(f'Trainable rows (non-NA label): {len(trainable):,}')
print(f'Positive rate:                 {trainable["y_fail_or_critical_next_180d"].mean():.2%}')

# Same number, broken out by year of anchor inspection. A flat line means the
# label is stationary; a sloped line is a heads-up that we may need
# year-aware features or a recency-weighted train set.
by_year = (
    trainable.assign(year=trainable['inspection_date'].dt.year)
    .groupby('year')['y_fail_or_critical_next_180d']
    .agg(['mean', 'size'])
    .rename(columns={'mean': 'positive_rate', 'size': 'n_rows'})
)
by_year['positive_rate'] = by_year['positive_rate'].astype(float).round(4)
by_year

### Modelable rows vs trainable labels

The raw inspection table contains rows that are not true pass/fail food-safety outcomes, and the label builder also masks burn-in rows and invalid-license rows. This check separates the broad modelable inspection universe from the final rows that have a usable forward-window label.

In [ ]:
MODELABLE_RESULTS = {"Pass", "Pass w/ Conditions", "Fail"}

modelable_vs_trainable = pd.Series({
    "raw_rows": len(raw),
    "modelable_result_rows": raw["results"].isin(MODELABLE_RESULTS).sum(),
    "labeled_rows": len(labeled),
    "trainable_label_rows": len(trainable),
    "modelable_result_share": raw["results"].isin(MODELABLE_RESULTS).mean(),
    "trainable_label_share": len(trainable) / len(labeled),
})

modelable_vs_trainable.to_frame("value")

In [ ]:
ax = by_year['positive_rate'].plot(
    title='P(y=1) by anchor year — should be roughly flat',
    marker='o', color='#15110D',
)
ax.set_ylabel('Positive rate'); ax.set_xlabel('')
ax.axhline(trainable['y_fail_or_critical_next_180d'].mean(), color='#B82E3A', linestyle=':',
           label='Overall positive rate')
ax.legend(); plt.tight_layout(); plt.show()

### Right truncation check

Rows near the end of the observed data may not have a full 180-day future window. This check shows whether right truncation is concentrated in the most recent anchor year, which is expected. If right truncation appeared throughout the historical period, that would suggest a label-window bug or data coverage problem.

In [ ]:
right_truncation_by_year = (
    labeled
    .assign(year=labeled["inspection_date"].dt.year)
    .groupby("year")
    .agg(
        n_rows=("inspection_id", "size"),
        right_truncated_share=("right_truncated", "mean"),
        labeled_share=("y_fail_or_critical_next_180d", lambda s: s.notna().mean()),
    )
)

right_truncation_by_year.tail(8).round(3)

In [ ]:
ax = right_truncation_by_year["right_truncated_share"].plot(
    marker="o",
    title="Right-truncated share by anchor year"
)
ax.set_xlabel("")
ax.set_ylabel("Share right-truncated")
plt.tight_layout()
plt.show()

### 5c. Sanity check — anchor result vs label

If the label encodes future events correctly, the anchor's *current* result
should be only weakly informative. A facility that just passed cleanly will
still sometimes be a positive (because something might fail later); a
facility that just failed will not automatically be a positive (because
*the anchor itself is excluded* — only what happens AFTER counts).

A bug we'd catch here: if the anchor-exclusion logic was wrong and the
anchor's own Fail leaked into its own label, `Fail` anchors would have
label rate near 100 %.

In [ ]:
anchor_vs_label = trainable.groupby('results').agg(
    n=('y_fail_or_critical_next_180d', 'size'),
    label_rate=('y_fail_or_critical_next_180d', lambda s: s.astype(float).mean()),
).round(3)
print('Label rate by anchor result:')
print(anchor_vs_label.sort_values('label_rate', ascending=False))

### 5d. Sanity check — repeat-failure rate at the facility level

Among facilities that have ever had a Fail, what fraction of their
(non-burnin, valid-license) inspections are positives? This should be
noticeably higher than the overall rate. If it isn't, that's a sign the
label isn't picking up the structure we care about.

In [ ]:
ever_failed = trainable.groupby('license_id')['is_fail_or_priority'].any()
facilities_with_any_event = ever_failed[ever_failed].index
label_rate_facility_with_event = trainable[
    trainable['license_id'].isin(facilities_with_any_event)
]['y_fail_or_critical_next_180d'].mean()
label_rate_facility_never_event = trainable[
    ~trainable['license_id'].isin(facilities_with_any_event)
]['y_fail_or_critical_next_180d'].mean()

print(f'Label rate at facilities with at least one event in record: {label_rate_facility_with_event:.2%}')
print(f'Label rate at facilities with no events in record:          {label_rate_facility_never_event:.2%}')

## Label construction summary for handoff

This final summary collects the main label health checks in one place before writing the contract artifact. These are the numbers downstream modeling should reference when explaining class balance, row eligibility, and censoring.

In [ ]:
label_health_summary = pd.Series({
    "total_labeled_rows": len(labeled),
    "trainable_rows": len(trainable),
    "positive_rate_trainable": trainable["y_fail_or_critical_next_180d"].mean(),
    "burnin_share": labeled["is_burnin"].mean(),
    "invalid_license_share": labeled["license_id"].isin({"0", ""}).mean(),
    "right_truncated_share": labeled["right_truncated"].mean(),
    "anchor_fail_label_rate": anchor_vs_label.loc["Fail", "label_rate"] if "Fail" in anchor_vs_label.index else pd.NA,
    "facility_with_event_label_rate": label_rate_facility_with_event,
    "facility_without_event_label_rate": label_rate_facility_never_event,
})

label_health_summary.to_frame("value").round(4)

**Label handoff takeaways**

- The label is forward-looking and excludes the anchor inspection itself.
- Burn-in rows and invalid-license rows are intentionally not treated as negatives.
- Right truncation should mostly affect the most recent rows because those rows do not have a full future observation window.
- The anchor `Fail` label rate should not be close to 100%; if it were, that would suggest leakage from the current inspection into the future label.
- Facilities with any prior bad event should have a higher future positive rate than facilities with no bad events.

## 6. Write the contract artifact

Outputs `data/processed/inspections_labeled.parquet`. Downstream consumers
(feature engineering in notebook 03, model training in 04–05) read from here.
Schema is documented in `docs/interface_contracts.md` § 1.

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
out_path = PROCESSED_DIR / 'inspections_labeled.parquet'

# Drop the temporary list-of-ints column. It's not in the contract; future
# notebooks that need violation codes call `add_violation_features` directly.
# We could keep it (parquet handles lists), but it adds 5-15 MB for no real
# downstream consumer.
to_write = labeled.copy()
if 'violation_codes' in to_write.columns:
    to_write = to_write.drop(columns=['violation_codes'])

# String-cast object columns to avoid pyarrow's mixed-type complaints.
obj_cols = to_write.select_dtypes('object').columns
to_write[obj_cols] = to_write[obj_cols].astype('string')

to_write.to_parquet(out_path, index=False)
size_mb = out_path.stat().st_size / 1e6
print(f'wrote → {out_path}')
print(f'        {len(to_write):,} rows · {to_write.shape[1]} cols · {size_mb:.1f} MB')
print()
print(f'Label prevalence (trainable rows): {trainable["y_fail_or_critical_next_180d"].mean():.2%}')

## 7. Hand-off

**Output**: `data/processed/inspections_labeled.parquet`  
**Schema**: see `docs/interface_contracts.md` § 1.

**Next step**: notebook `03_feature_engineering.ipynb` reads this file, joins
to business licenses + 311, builds leak-free `prior_*` features (every
`prior_*` column MUST use `.shift()` or `< as_of_date` guards — see
CLAUDE.md), and writes `data/processed/features.parquet`.

**Sanity to confirm before moving on**:

- Trainable rows ≥ ~150,000 (yes if data range covers 2019-onward)
- Label prevalence in the 10–40 % range
- Year-over-year positive-rate plot is roughly flat (no glaring drift)
- Label rate higher at facilities with prior events than at those without
- `tests/test_labels.py` passes (it does — 16/16 as of 2026-06-02)